# 01 – Bronze Layer: EMS Incident Dispatch Ingestion

**Purpose:** Ingest the raw NYC EMS incident dispatch data into the Bronze layer exactly as it was received, with metadata that makes every snapshot traceable.

**Source:** NYC Open Data – EMS Incident Dispatch Data (`76xm-jjuj`)

**Bronze rules:** no cleaning, no type changes, no overwriting of earlier snapshots.

## 1. Load configuration

Paths and settings are stored in `config/dev.yaml` and `config/prod.yaml`, not in the code. Change `ENV` to `"prod"` to run on the full dataset.

In [1]:
import sys
sys.path.append("../src")          # lets the notebook import files from src/

from config_loader import load_config

ENV = "prod"                        # change to "prod" for the full data
config = load_config(f"../config/{ENV}.yaml")

print(config)

{'environment': 'prod', 'source': {'dataset_name': 'ems_incident_dispatch', 'dataset_id': '76xm-jjuj', 'input_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data_sample/EMS_Incident_Dispatch_Data.csv'}, 'bronze': {'root_path': '/home/ubuntu/spark-notebooks/ems-pipeline/data/bronze'}}


## 2. Start Spark

Create the Spark session, which is the entry point for all Spark operations. `getOrCreate()` reuses an existing session if one is already running. The printed version confirms Spark is working.

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ems_bronze_ingest")
    .getOrCreate()
)

print(spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/02 15:16:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/02 15:16:11 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/02 15:16:11 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


3.5.0


## 3. Read the raw CSV

The file is read with `inferSchema=False`, so every column stays as text. Bronze keeps data in its original form, and type conversion happens later in the Silver layer.

**Sanity check:** the sample should have 2,000 rows and 31 columns.

In [3]:
import os

input_path = os.path.join("..", config["source"]["input_path"])

raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(input_path)
)

print("Rows:", raw_df.count())
print("Columns:", len(raw_df.columns))

Rows: 30389767
Columns: 31


## 4. Inspect the raw data

Check the schema (all columns should be `string`) and look at a few rows to see what the raw values look like, such as dates stored as text and any empty fields.

In [4]:
raw_df.printSchema()
raw_df.show(5, truncate=False)

26/11/02 15:16:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


root
 |-- INCIDENT_ID: string (nullable = true)
 |-- INCIDENT_DATETIME: string (nullable = true)
 |-- INITIAL_CALL_TYPE: string (nullable = true)
 |-- INITIAL_SEVERITY_LEVEL_CODE: string (nullable = true)
 |-- FINAL_CALL_TYPE: string (nullable = true)
 |-- FINAL_SEVERITY_LEVEL_CODE: string (nullable = true)
 |-- FIRST_ASSIGNMENT_DATETIME: string (nullable = true)
 |-- VALID_DISPATCH_RSPNS_TIME_INDC: string (nullable = true)
 |-- DISPATCH_RESPONSE_SECONDS_QY: string (nullable = true)
 |-- FIRST_ACTIVATION_DATETIME: string (nullable = true)
 |-- FIRST_ON_SCENE_DATETIME: string (nullable = true)
 |-- VALID_INCIDENT_RSPNS_TIME_INDC: string (nullable = true)
 |-- INCIDENT_RESPONSE_SECONDS_QY: string (nullable = true)
 |-- INCIDENT_TRAVEL_TM_SECONDS_QY: string (nullable = true)
 |-- FIRST_TO_HOSP_DATETIME: string (nullable = true)
 |-- FIRST_HOSP_ARRIVAL_DATETIME: string (nullable = true)
 |-- INCIDENT_CLOSE_DATETIME: string (nullable = true)
 |-- HELD_INDICATOR: string (nullable = true)
 |-

## 5. Add traceability metadata

Bronze keeps all original columns unchanged. We add four metadata columns so every row can be traced back to its source:

- `_ingestion_timestamp`: when the data was loaded
- `_run_id`: a unique ID for this pipeline run
- `_source_file`: the file the row came from
- `_dataset_id`: the NYC Open Data dataset ID

We also create a `snapshot_date`, which will name the Bronze folder in the next step.

In [5]:
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F

run_id = str(uuid.uuid4())
snapshot_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

bronze_df = (
    raw_df
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_run_id", F.lit(run_id))
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_dataset_id", F.lit(config["source"]["dataset_id"]))
)

print("Run ID:", run_id)
print("Snapshot date:", snapshot_date)
print("Columns now:", len(bronze_df.columns))

bronze_df.select(
    "INCIDENT_ID", "_ingestion_timestamp", "_run_id", "_source_file", "_dataset_id"
).show(3, truncate=False)

Run ID: d5fcf808-1500-416f-8e1e-b176e0744c48
Snapshot date: 2026-11-02
Columns now: 35
+-----------+-----------------------+------------------------------------+-------------------------------------------------------------------------------------------+-----------+
|INCIDENT_ID|_ingestion_timestamp   |_run_id                             |_source_file                                                                               |_dataset_id|
+-----------+-----------------------+------------------------------------+-------------------------------------------------------------------------------------------+-----------+
|16975942   |2026-11-02 15:16:22.286|d5fcf808-1500-416f-8e1e-b176e0744c48|file:///home/ubuntu/spark-notebooks/ems-pipeline/data_sample/EMS_Incident_Dispatch_Data.csv|76xm-jjuj  |
|16975943   |2026-11-02 15:16:22.286|d5fcf808-1500-416f-8e1e-b176e0744c48|file:///home/ubuntu/spark-notebooks/ems-pipeline/data_sample/EMS_Incident_Dispatch_Data.csv|76xm-jjuj  |
|16975944   |2026-

## 6. Write the Bronze snapshot

The data is saved as Parquet in `bronze/<dataset_name>/snapshot_date=<date>/`.

- All columns are still strings, so the data remains raw. Parquet only changes the storage format.
- If a snapshot for this date already exists, the write is **skipped**. Bronze never overwrites earlier snapshots.

In [6]:
bronze_path = os.path.abspath(os.path.join(
    "..",
    config["bronze"]["root_path"],
    config["source"]["dataset_name"],
    f"snapshot_date={snapshot_date}",
))

print("Bronze path:", bronze_path)

if os.path.exists(bronze_path):
    print("Snapshot already exists - skipping write.")
else:
    bronze_df.write.mode("errorifexists").parquet(bronze_path)
    print("Snapshot written.")

Bronze path: /home/ubuntu/spark-notebooks/ems-pipeline/data/bronze/ems_incident_dispatch/snapshot_date=2026-11-02


26/11/02 15:16:23 WARN MemoryManager: Total allocation exceeds 95.00% (906,992,014 bytes) of heap memory
Scaling row group sizes to 96.54% for 7 writers
26/11/02 15:16:23 WARN MemoryManager: Total allocation exceeds 95.00% (906,992,014 bytes) of heap memory
Scaling row group sizes to 84.47% for 8 writers
26/11/02 15:16:35 WARN MemoryManager: Total allocation exceeds 95.00% (906,992,014 bytes) of heap memory
Scaling row group sizes to 96.54% for 7 writers
26/11/02 15:16:35 WARN MemoryManager: Total allocation exceeds 95.00% (906,992,014 bytes) of heap memory
Scaling row group sizes to 96.54% for 7 writers
26/11/02 15:16:35 WARN MemoryManager: Total allocation exceeds 95.00% (906,992,014 bytes) of heap memory
Scaling row group sizes to 84.47% for 8 writers
26/11/02 15:16:35 WARN MemoryManager: Total allocation exceeds 95.00% (906,992,014 bytes) of heap memory
Scaling row group sizes to 96.54% for 7 writers
26/11/02 15:16:35 WARN MemoryManager: Total allocation exceeds 95.00% (906,992,014

Snapshot written.


### Verify the snapshot

Read the Parquet files back and compare them with the original input. The row count should match, and all columns should still be strings.

In [7]:
check_df = spark.read.parquet(bronze_path)

print("Rows written:", check_df.count())
print("Rows in raw input:", raw_df.count())
print("Columns:", len(check_df.columns))

check_df.printSchema()
print(os.listdir(bronze_path))

Rows written: 30389767


Rows in raw input: 30389767
Columns: 35
root
 |-- INCIDENT_ID: string (nullable = true)
 |-- INCIDENT_DATETIME: string (nullable = true)
 |-- INITIAL_CALL_TYPE: string (nullable = true)
 |-- INITIAL_SEVERITY_LEVEL_CODE: string (nullable = true)
 |-- FINAL_CALL_TYPE: string (nullable = true)
 |-- FINAL_SEVERITY_LEVEL_CODE: string (nullable = true)
 |-- FIRST_ASSIGNMENT_DATETIME: string (nullable = true)
 |-- VALID_DISPATCH_RSPNS_TIME_INDC: string (nullable = true)
 |-- DISPATCH_RESPONSE_SECONDS_QY: string (nullable = true)
 |-- FIRST_ACTIVATION_DATETIME: string (nullable = true)
 |-- FIRST_ON_SCENE_DATETIME: string (nullable = true)
 |-- VALID_INCIDENT_RSPNS_TIME_INDC: string (nullable = true)
 |-- INCIDENT_RESPONSE_SECONDS_QY: string (nullable = true)
 |-- INCIDENT_TRAVEL_TM_SECONDS_QY: string (nullable = true)
 |-- FIRST_TO_HOSP_DATETIME: string (nullable = true)
 |-- FIRST_HOSP_ARRIVAL_DATETIME: string (nullable = true)
 |-- INCIDENT_CLOSE_DATETIME: string (nullable = true)
 |-- HELD

## 7. Record the run metadata

A small JSON log is written for every ingestion run. It records what was loaded, when, from where, how many rows, and which columns. This makes each snapshot auditable and lets future runs detect schema changes by comparing column lists.

Logs are stored in `bronze/_ingestion_log/`, separate from the data snapshots.

In [8]:
import json

log_dir = os.path.abspath(os.path.join("..", config["bronze"]["root_path"], "_ingestion_log"))
os.makedirs(log_dir, exist_ok=True)

run_log = {
    "run_id": run_id,
    "environment": config["environment"],
    "dataset_name": config["source"]["dataset_name"],
    "dataset_id": config["source"]["dataset_id"],
    "snapshot_date": snapshot_date,
    "ingested_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_path": config["source"]["input_path"],
    "bronze_path": bronze_path,
    "row_count": check_df.count(),
    "column_count": len(check_df.columns),
    "columns": check_df.columns,
    "status": "success",
}

log_file = os.path.join(log_dir, f"{config['source']['dataset_name']}_{snapshot_date}_{run_id}.json")

with open(log_file, "w") as f:
    json.dump(run_log, f, indent=2)

print("Log written to:", log_file)
print(json.dumps({k: v for k, v in run_log.items() if k != "columns"}, indent=2))

Log written to: /home/ubuntu/spark-notebooks/ems-pipeline/data/bronze/_ingestion_log/ems_incident_dispatch_2026-11-02_d5fcf808-1500-416f-8e1e-b176e0744c48.json
{
  "run_id": "d5fcf808-1500-416f-8e1e-b176e0744c48",
  "environment": "prod",
  "dataset_name": "ems_incident_dispatch",
  "dataset_id": "76xm-jjuj",
  "snapshot_date": "2026-11-02",
  "ingested_at_utc": "2026-11-02T15:17:42.863189+00:00",
  "source_path": "/home/ubuntu/spark-notebooks/ems-pipeline/data_sample/EMS_Incident_Dispatch_Data.csv",
  "bronze_path": "/home/ubuntu/spark-notebooks/ems-pipeline/data/bronze/ems_incident_dispatch/snapshot_date=2026-11-02",
  "row_count": 30389767,
  "column_count": 35,
  "status": "success"
}
